In [1]:
from pathlib import Path
import os
import sys

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'config/paper.json').is_file())
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score, matthews_corrcoef
from IPython.display import display
n_bootstrap = 10000
block_length = 21
confidence_level = 0.95
random_seed = 42
output = Path('Benchmark/Integrated')

folder = Path('FINDER/Official')
scores = pd.read_csv(folder / 'daily_scores_all_markets.csv')
metrics = pd.read_csv(folder / 'market_agent_metrics.csv')
thresholds = pd.read_csv(folder / 'daily_thresholds.csv')
rows = []
for (market, agent), frame in thresholds.groupby(['market', 'agent']):
    source = scores[scores.market.eq(market)].set_index('target_date').loc[frame.target_date]
    actual = source.actual_label.to_numpy()
    probability = source[agent].to_numpy()
    predicted = (probability >= frame.threshold.to_numpy()).astype(int)
    assert np.array_equal(predicted, frame.prediction)
    row = {'market': market, 'agent': agent, 'roc_auc': roc_auc_score(actual, probability), 'ap': average_precision_score(actual, probability), 'precision': precision_score(actual, predicted, zero_division=0), 'recall': recall_score(actual, predicted, zero_division=0), 'f1': f1_score(actual, predicted, zero_division=0), 'mcc': matthews_corrcoef(actual, predicted)}
    expected = metrics[metrics.market.eq(market) & metrics.agent.eq(agent)].iloc[0]
    for field in ['roc_auc', 'ap', 'precision', 'recall', 'f1', 'mcc']:
        assert np.isclose(row[field], expected[field])
    rows.append(row)
calculated = pd.DataFrame(rows)
display(calculated)


     market                agent   roc_auc  ...    recall        f1       mcc
0      CACT         chart_expert  0.583465  ...  0.190476  0.148148  0.089538
1      CACT  decision_supervisor  0.676072  ...  0.285714  0.206897  0.154179
2      CACT          news_expert  0.665417  ...  0.428571  0.211765  0.168807
3      CACT          news_reader  0.661668  ...  0.285714  0.200000  0.146451
4      CACT  quantitative_expert  0.698303  ...  0.190476  0.216216  0.179009
5    CBU0.L         chart_expert  0.570330  ...  0.133333  0.067797  0.010924
6    CBU0.L  decision_supervisor  0.659066  ...  0.333333  0.158730  0.126156
7    CBU0.L          news_expert  0.640018  ...  0.266667  0.150943  0.112467
8    CBU0.L          news_reader  0.761081  ...  0.200000  0.153846  0.113921
9    CBU0.L  quantitative_expert  0.592125  ...  0.266667  0.150943  0.112467
10  FTSE100         chart_expert  0.614215  ...  0.296296  0.195122  0.118873
11  FTSE100  decision_supervisor  0.677294  ...  0.296296  0.242

In [2]:
from finder_core.public_outputs import sequential_thresholds
from finder_core.benchmark_evaluation import moving_block_intervals
import hashlib
import json

markets = {'^GSPC': 'SP500', 'FTSE100': 'FTSE', 'GDAXI': 'DAX', 'CACT': 'CAC', 'STOXX': 'STOXX', 'CBU0.L': 'CBU0'}
finder_models = ['decision_supervisor', 'quantitative_expert', 'chart_expert', 'news_reader', 'news_expert']
models = finder_models + ['logistic_ind', 'lstm_ind', 'logistic_news', 'lstm_news', 'gjr_garch']
source_root = Path('Benchmark/Results/Benchmark_Outputs')
frames, coverage, source_manifest, label_differences = [], [], [], []
for market, code in markets.items():
    frame = scores.loc[scores.market.eq(market), ['target_date', 'information_cutoff', 'actual_label', *finder_models]].copy()
    original_n = len(frame)
    for folder, suffix, variant in [('Without_news_v2', 'finderBenchmark', 'ind'), ('With_news_v2', 'finderBenchmark', 'news'), ('GJR_GARCH', 'GJR_GARCH', 'garch')]:
        path = source_root / folder / f'{code}_{suffix}_test_predictions.csv'
        source_manifest.append({'path': str(path), 'sha256': hashlib.sha256(path.read_bytes()).hexdigest()})
        benchmark = pd.read_csv(path)
        columns = {'garch_probability': 'gjr_garch'} if variant == 'garch' else {'Logistic_probability': f'logistic_{variant}', 'LSTM_probability': f'lstm_{variant}'}
        frame = frame.merge(benchmark[['date', 'actual', *columns]], left_on='information_cutoff', right_on='date', validate='one_to_one')
        differences = frame.loc[frame.actual_label.ne(frame.actual), ['target_date', 'information_cutoff', 'actual_label', 'actual']].assign(market=market, variant=variant)
        label_differences.append(differences)
        frame = frame.drop(columns=['date', 'actual']).rename(columns=columns)
    frame = frame.sort_values('target_date').reset_index(drop=True)
    assert np.isfinite(frame[models]).all().all() and not frame.target_date.duplicated().any()
    frame['market'] = market
    coverage.append({'market': market, 'finder_n': original_n, 'matched_n': len(frame), 'excluded_n': original_n - len(frame), 'events': int(frame.actual_label.sum())})
    frames.append(frame)
display(pd.DataFrame(coverage))


    market  finder_n  matched_n  excluded_n  events
0    ^GSPC       374        374           0      17
1  FTSE100       379        377           2      27
2    GDAXI       379        379           0      21
3     CACT       383        381           2      21
4    STOXX       385        383           2      22
5   CBU0.L       379        377           2      15


In [3]:
metric_rows, difference_rows, decision_frames = [], [], []
settings = {'method': 'moving_block_percentile', 'n_bootstrap': n_bootstrap, 'block_length': block_length, 'confidence_level': confidence_level, 'random_seed': random_seed, 'block_unit': 'retained_market_observations'}
for frame in frames:
    market = frame.market.iloc[0]
    y = frame.actual_label.to_numpy()
    draws = moving_block_intervals(frame, models, n_bootstrap, block_length, random_seed)
    points = np.array([[roc_auc_score(y, frame[model]), average_precision_score(y, frame[model])] for model in models])
    for j, model in enumerate(models):
        thresholds = sequential_thresholds(frame[model], y, frame.target_date, frame.information_cutoff)
        predicted = (frame[model].to_numpy() >= thresholds).astype(int)
        decision_frames.append(frame[['market', 'target_date', 'information_cutoff', 'actual_label']].assign(model=model, probability=frame[model], threshold=thresholds, prediction=predicted))
        classification = {'precision': precision_score(y, predicted, zero_division=0), 'recall': recall_score(y, predicted, zero_division=0), 'f1': f1_score(y, predicted, zero_division=0), 'mcc': matthews_corrcoef(y, predicted)}
        for k, metric in enumerate(['roc_auc', 'ap']):
            bounds = np.quantile(draws[:, j, k], [(1 - confidence_level) / 2, (1 + confidence_level) / 2])
            metric_rows.append({'market': market, 'model': model, 'metric': metric, 'n': len(y), 'events': int(y.sum()), 'estimate': points[j, k], 'ci_lower': bounds[0], 'ci_upper': bounds[1], **classification, **settings})
            if j:
                bounds = np.quantile(draws[:, 0, k] - draws[:, j, k], [(1 - confidence_level) / 2, (1 + confidence_level) / 2])
                difference_rows.append({'market': market, 'comparator': model, 'metric': metric, 'estimate': points[0, k] - points[j, k], 'ci_lower': bounds[0], 'ci_upper': bounds[1], **settings})
    print(market, len(frame), 'complete')


^GSPC 374 complete
FTSE100 377 complete
GDAXI 379 complete
CACT 381 complete
STOXX 383 complete
CBU0.L 377 complete


In [4]:
metrics_ci = pd.DataFrame(metric_rows)
paired_differences_ci = pd.DataFrame(difference_rows)
output.mkdir(parents=True, exist_ok=True)
metrics_ci.to_csv(output / 'metrics_ci.csv', index=False)
paired_differences_ci.to_csv(output / 'paired_differences_ci.csv', index=False)
pd.concat(decision_frames, ignore_index=True).to_csv(output / 'daily_decisions.csv', index=False)
pd.DataFrame(coverage).to_csv(output / 'coverage.csv', index=False)
(output / 'source_manifest.json').write_text(json.dumps(source_manifest, indent=2) + '\n')
display(metrics_ci.query("metric == 'roc_auc'").pivot(index='market', columns='model', values='estimate').round(4))
pd.concat(label_differences, ignore_index=True).to_csv(output / 'source_label_differences.csv', index=False)


model    chart_expert  decision_supervisor  ...  news_reader  quantitative_expert
market                                      ...                                  
CACT           0.5838               0.6754  ...       0.6603               0.6986
CBU0.L         0.5685               0.6581  ...       0.7605               0.5910
FTSE100        0.6137               0.6763  ...       0.6417               0.7093
GDAXI          0.5948               0.5924  ...       0.4125               0.6996
STOXX          0.6222               0.6443  ...       0.5574               0.6509
^GSPC          0.7060               0.7107  ...       0.6289               0.7121

[6 rows x 10 columns]


In [5]:
evaluate_new_run = False
run_id = 'concept'
ticker = '^GSPC'

if evaluate_new_run:
    from finder_core.settings import load_settings
    from finder_core.public_outputs import evaluate
    settings = load_settings(root, ticker=ticker)
    scores, metrics = evaluate(root / 'Runs' / ticker / run_id, root / settings['paths']['labels'])
    display(metrics)


In [6]:
import matplotlib.pyplot as plt

plot_models = ['decision_supervisor', 'lstm_ind', 'lstm_news', 'logistic_ind', 'logistic_news', 'gjr_garch']
for metric in ['roc_auc', 'ap']:
    plt.figure(figsize=(12, 9))
    for i, market in enumerate(markets):
        plt.subplot(3, 2, i + 1)
        values = metrics_ci.query('market == @market and metric == @metric').set_index('model').loc[plot_models]
        positions = np.arange(len(values))
        plt.hlines(positions, values.ci_lower, values.ci_upper, color='gray')
        plt.scatter(values.estimate, positions, color=['royalblue', 'gray', 'gray', 'black', 'black', 'salmon'])
        plt.yticks(positions, plot_models, fontsize=8)
        plt.xlabel(metric.upper())
        plt.title(market)
        plt.xlim(0, 1)
    plt.tight_layout()
    plt.savefig(output / f'{metric}_comparison.png', dpi=180)
    plt.close()
